In [1]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils import data
torch.manual_seed(42)

epoch = 300
batch_size = 64
num_steps = 20
lr = 0.1
num_examples = 600

data_path = "../data/fra-eng/fra.txt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

device: cuda


In [2]:
def data_read(path: str) -> str:
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def preprocess_nmt(text: str) -> str:
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )

    result = []

    for i, char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(" ")

        result.append(char)

    return "".join(result)


def tokenize_nmt(text: str, max_text: int = 600):
    source = []
    target = []

    for line in text.splitlines():
        parts = line.split("\t")

        if len(parts) < 2:
            continue

        source.append(parts[0].split())
        target.append(parts[1].split())

        if max_text is not None and len(source) >= max_text:
            break

    return source, target

In [3]:
def vocab_maker(tokens, min_freq: int = 2):
    counter = Counter()

    for line in tokens:
        counter.update(line)

    token_freq = sorted(
        counter.items(),
        key=lambda item: (-item[1], item[0])
    )

    idx_to_token = [
        "<unk>",
        "<pad>",
        "<bos>",
        "<eos>",
    ]

    for token, freq in token_freq:
        if freq < min_freq:
            continue

        if token not in idx_to_token:
            idx_to_token.append(token)

    token_to_idx = {
        token: idx
        for idx, token in enumerate(idx_to_token)
    }

    return {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx,
    }


def truncate_pad(line, step_num: int, pad_idx: int):
    if len(line) > step_num:
        return line[:step_num]

    return line + [pad_idx] * (step_num - len(line))


def build_array_nmt(lines, vocab, step_num: int):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        line_idx.append(eos_idx)
        line_idx = truncate_pad(
            line_idx,
            step_num,
            pad_idx,
        )

        array.append(line_idx)

    array = torch.tensor(array, dtype=torch.long)
    valid_len = (array != pad_idx).sum(dim=1)

    return array, valid_len


def load_data_nmt(
    batch_size: int,
    step_num: int,
    max_examples: int = 600,
):
    raw_text = data_read(data_path)
    text = preprocess_nmt(raw_text)
    source, target = tokenize_nmt(text, max_examples)

    src_vocab = vocab_maker(source, min_freq=2)
    tgt_vocab = vocab_maker(target, min_freq=2)

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        step_num,
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        step_num,
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len,
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    return train_iter, src_vocab, tgt_vocab

In [4]:
train_iter, src_vocab, tgt_vocab = load_data_nmt(
    batch_size,
    num_steps,
    num_examples,
)

src_pad_idx = src_vocab["token_to_idx"]["<pad>"]
tgt_pad_idx = tgt_vocab["token_to_idx"]["<pad>"]
tgt_bos_idx = tgt_vocab["token_to_idx"]["<bos>"]
tgt_eos_idx = tgt_vocab["token_to_idx"]["<eos>"]

X, X_valid_len, Y, Y_valid_len = next(iter(train_iter))

print("X shape:", X.shape)
print("X_valid_len shape:", X_valid_len.shape)
print("Y shape:", Y.shape)
print("Y_valid_len shape:", Y_valid_len.shape)
print("src vocab size:", len(src_vocab["idx_to_token"]))
print("tgt vocab size:", len(tgt_vocab["idx_to_token"]))

X shape: torch.Size([64, 20])
X_valid_len shape: torch.Size([64])
Y shape: torch.Size([64, 20])
Y_valid_len shape: torch.Size([64])
src vocab size: 184
tgt vocab size: 201


In [5]:
def mask_softmax(
    scores: torch.Tensor,
    valid_len: torch.Tensor,
) -> torch.Tensor:
    # scores:    (batch, query_num, key_num)
    # valid_len: (batch,)

    key_num = scores.shape[-1]

    positions = torch.arange(
        key_num,
        device=scores.device,
    ).reshape(1, 1, -1)

    mask = positions >= valid_len.reshape(-1, 1, 1)
    scores = scores.masked_fill(mask, -1e6)

    return torch.softmax(scores, dim=-1)

In [6]:
class attention_module(nn.Module):
    def __init__(self,query_size,key_size,value_size,attention_hidden,head_num):
        super().__init__()
        self.head_num=head_num
        self.head_size=attention_hidden//head_num
        self.w_q=nn.Linear(query_size,attention_hidden)
        self.w_k=nn.Linear(key_size,attention_hidden)
        self.w_v=nn.Linear(value_size,attention_hidden)
        self.w_o=nn.Linear(attention_hidden,attention_hidden)
    def split_head(self,x):
        # x (batch,step,attention_hidden)
        batch=x.shape[0]
        step=x.shape[1]
        x=x.reshape(batch,step,self.head_num,self.head_size)
        x=x.permute(0,2,1,3)
        return x.reshape(-1,step,self.head_size) #batch*head step head_size
    def merge_head(self,x):
        batch=x.shape[0]//self.head_num
        step=x.shape[1]
        x=x.reshape(batch,self.head_num,step,self.head_size)
        x=x.permute(0,2,1,3)
        return x.reshape(batch,step,-1)
    def forward(self,query,key,value,valid_len):
        #query batch 1 decoder_hidden
        #key batch step encoder_output_size
        query=self.w_q(query)
        key=self.w_k(key)
        value=self.w_v(value)
        query=self.split_head(query)
        key=self.split_head(key)
        value=self.split_head(value)
        attention_weight=torch.bmm(query,key.transpose(1,2))/math.sqrt(self.head_size)
        valid_len=valid_len.repeat_interleave(self.head_num)
        attention_weight=mask_softmax(attention_weight,valid_len)
        score=torch.bmm(attention_weight,value)
        score=self.merge_head(score)
        return self.w_o(score)

class encoder(nn.Module):
    def __init__(self,input_num,embedding_num,hidden_num,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.embedding_num=embedding_num
        self.hidden_num=hidden_num
        self.embedding=nn.Embedding(input_num,embedding_num,padding_idx=pad_idx)
        self.gru=nn.GRU(embedding_num,hidden_num,bidirectional=True)
    def forward(self,x):
        x=self.embedding(x)
        x=x.permute(1,0,2)
        output,state=self.gru(x)
        return output,state
class decoder(nn.Module):
    def __init__(self,input_num,embedding_num,hidden_num,pad_idx,attention_hidden,encoder_output_size):
        super().__init__()
        self.input_num=input_num
        self.embedding_num=embedding_num
        self.hidden_num=hidden_num
        self.embedding=nn.Embedding(input_num,embedding_num,padding_idx=pad_idx)
        self.attention_module=attention_module(hidden_num,encoder_output_size,encoder_output_size,attention_hidden,8)
        self.gru=nn.GRU(embedding_num+attention_hidden,hidden_num)
        self.linear=nn.Linear(hidden_num,input_num)
    def forward(self,y,encoder_output,encoder_state,valid_len):
        y=self.embedding(y)
        y=y.permute(1,0,2)
        outputs=[]
        for i in y:
            #i batch embedding
            query=encoder_state[-1].unsqueeze(1) #batch 1 decoder_hidden_num
            #key batch step encoder_output_size
            context=self.attention_module(query,encoder_output,encoder_output,valid_len) #batch 1 attention_hidden
            i=i.unsqueeze(1)
            input=torch.cat((i,context),dim=2)
            input=input.permute(1,0,2)
            output,encoder_state=self.gru(input,encoder_state)
            outputs.append(output)
        outputs=torch.cat(outputs,dim=0)
        outputs=outputs.reshape(-1,self.hidden_num)
        outputs=self.linear(outputs)
        return outputs,encoder_state
class seq2seq(nn.Module):
    def __init__(self,encoder,decoder,bos_idx):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
        self.bos_idx=bos_idx
        self.linear=nn.Linear(encoder.hidden_num*2,decoder.hidden_num)
    def forward(self,x,x_len,y):
        encoder_output,encoder_state=self.encoder(x)
        #encoder_output step batch encoder_output_size
        state1=encoder_state[0]
        state2=encoder_state[1]
        state=torch.cat((state1,state2),dim=1) #batch  encoder_output_size
        state=self.linear(state)
        state=torch.tanh(state)
        state=state.unsqueeze(0).repeat(self.decoder.gru.num_layers,1,1) 
        encoder_output=encoder_output.permute(1,0,2)
        bos=torch.full(size=(y.shape[0],1),fill_value=self.bos_idx,dtype=torch.long,device=y.device)
        y=torch.cat((bos,y[:,:-1]),dim=1)
        output,state=self.decoder(y,encoder_output,state,x_len)
        return output,state

myencoder=encoder(len(src_vocab['token_to_idx']),128,256,src_vocab['token_to_idx']['<pad>'])
mydecoder=decoder(len(tgt_vocab['token_to_idx']),256,512,tgt_vocab['token_to_idx']['<pad>'],256,512)
net=seq2seq(myencoder,mydecoder,tgt_vocab['token_to_idx']['<bos>']).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)

loss = nn.CrossEntropyLoss(
    ignore_index=tgt_pad_idx
)





In [7]:
for i in range(epoch):
    net.train()
    train_loss = 0.0
    train_token_count = 0
    for x,x_len,y,y_len in train_iter:
        x=x.to(device)
        y=y.to(device)
        x_len=x_len.to(device)
        trainer.zero_grad()
        logit,state=net(x,x_len,y)
        y=y.T.reshape(-1)
        l=loss(logit,y)
        l.backward()
        nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0,
        )
        trainer.step()
        valid_token_num = y_len.sum().item()
        train_loss += l.item() * valid_token_num
        train_token_count += valid_token_num

    average_loss = train_loss / train_token_count
    print(f"epoch{i},loss:{average_loss:.5f}")



epoch0,loss:4.73869
epoch1,loss:3.40073
epoch2,loss:2.87094
epoch3,loss:2.69135
epoch4,loss:2.57942
epoch5,loss:2.49936
epoch6,loss:2.43669
epoch7,loss:2.38779
epoch8,loss:2.34935
epoch9,loss:2.31354
epoch10,loss:2.28022
epoch11,loss:2.25049
epoch12,loss:2.22256
epoch13,loss:2.19655
epoch14,loss:2.16951
epoch15,loss:2.14277
epoch16,loss:2.11625
epoch17,loss:2.09505
epoch18,loss:2.06736
epoch19,loss:2.04209
epoch20,loss:2.02091
epoch21,loss:1.99722
epoch22,loss:1.97521
epoch23,loss:1.95580
epoch24,loss:1.93291
epoch25,loss:1.91329
epoch26,loss:1.89179
epoch27,loss:1.87458
epoch28,loss:1.85608
epoch29,loss:1.83777
epoch30,loss:1.81991
epoch31,loss:1.80182
epoch32,loss:1.78650
epoch33,loss:1.76998
epoch34,loss:1.75215
epoch35,loss:1.73748
epoch36,loss:1.72368
epoch37,loss:1.70366
epoch38,loss:1.68810
epoch39,loss:1.67429
epoch40,loss:1.65760
epoch41,loss:1.64483
epoch42,loss:1.62933
epoch43,loss:1.61403
epoch44,loss:1.60498
epoch45,loss:1.58676
epoch46,loss:1.57764
epoch47,loss:1.55859
ep

In [8]:
def predict_seq2seq(sentence, net, src_vocab, tgt_vocab, num_steps, device):
    net.eval()

    src_token_to_idx = src_vocab["token_to_idx"]
    tgt_token_to_idx = tgt_vocab["token_to_idx"]
    tgt_idx_to_token = tgt_vocab["idx_to_token"]

    src_unk_idx = src_token_to_idx["<unk>"]
    src_pad_idx = src_token_to_idx["<pad>"]
    src_eos_idx = src_token_to_idx["<eos>"]
    tgt_bos_idx = tgt_token_to_idx["<bos>"]
    tgt_eos_idx = tgt_token_to_idx["<eos>"]

    sentence = preprocess_nmt(sentence)
    source_tokens = sentence.split()
    source_ids = [
        src_token_to_idx.get(token, src_unk_idx)
        for token in source_tokens
    ]
    source_ids.append(src_eos_idx)

    source_valid_len = min(len(source_ids), num_steps)
    source_ids = truncate_pad(
        source_ids,
        num_steps,
        src_pad_idx,
    )

    x = torch.tensor(
        source_ids,
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)
    x_len = torch.tensor(
        [source_valid_len],
        dtype=torch.long,
        device=device,
    )

    result = []

    with torch.no_grad():
        encoder_output, encoder_state = net.encoder(x)
        encoder_output = encoder_output.permute(1, 0, 2)

        forward_state = encoder_state[0]
        backward_state = encoder_state[1]
        decoder_state = torch.cat(
            (forward_state, backward_state),
            dim=1,
        )
        decoder_state = torch.tanh(
            net.linear(decoder_state)
        )
        decoder_state = decoder_state.unsqueeze(0).repeat(
            net.decoder.gru.num_layers,
            1,
            1,
        )

        decoder_x = torch.tensor(
            [[tgt_bos_idx]],
            dtype=torch.long,
            device=device,
        )

        for _ in range(num_steps):
            logits, decoder_state = net.decoder(
                decoder_x,
                encoder_output,
                decoder_state,
                x_len,
            )

            predicted_idx = logits.argmax(dim=1).item()

            if predicted_idx == tgt_eos_idx:
                break

            result.append(tgt_idx_to_token[predicted_idx])

            decoder_x = torch.tensor(
                [[predicted_idx]],
                dtype=torch.long,
                device=device,
            )

    return " ".join(result)


translation = predict_seq2seq(
    "go .",
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device,
)
print(translation)


va !
